In [1]:
import json
import random
from datetime import datetime, timedelta
from ucimlrepo import fetch_ucirepo, list_available_datasets
import numpy as np
import pandas as pd
from etl import UserGenerator
from feature_engineer import FeatureEngineer
from sklearn.linear_model import LogisticRegression
from train_mlflow import TrainMlflow
from train_mlflow_advance import TrainOptuna

c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# ETL

In [2]:
user_generator = UserGenerator(n_samples=25000)


In [3]:
ds = user_generator.create_dataset()
print(type(ds), isinstance(ds, tuple))

<class 'pandas.core.frame.DataFrame'> False


In [4]:
ds

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...
541904,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


In [5]:
ds = user_generator.run_etl()

In [6]:
ds.info

<bound method DataFrame.info of                                 Description  Quantity         InvoiceDate  \
0        WHITE HANGING HEART T-LIGHT HOLDER         6 2010-12-01 08:26:00   
1                       WHITE METAL LANTERN         6 2010-12-01 08:26:00   
2            CREAM CUPID HEARTS COAT HANGER         8 2010-12-01 08:26:00   
3       KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-12-01 08:26:00   
4            RED WOOLLY HOTTIE WHITE HEART.         6 2010-12-01 08:26:00   
...                                     ...       ...                 ...   
541904          PACK OF 20 SPACEBOY NAPKINS        12 2011-12-09 12:50:00   
541905         CHILDREN'S APRON DOLLY GIRL          6 2011-12-09 12:50:00   
541906        CHILDRENS CUTLERY DOLLY GIRL          4 2011-12-09 12:50:00   
541907      CHILDRENS CUTLERY CIRCUS PARADE         4 2011-12-09 12:50:00   
541908        BAKING SET 9 PIECE RETROSPOT          3 2011-12-09 12:50:00   

        UnitPrice  CustomerID         Count

In [7]:
ds.describe().T

,count,mean,min,25%,50%,75%,max,std
Quantity,397884.0,12.988238,1.0,2.0,6.0,12.0,80995.0,179.331775
InvoiceDate,397884,2011-07-10 23:41:23.511023360,2010-12-01 08:26:00,2011-04-07 11:12:00,2011-07-31 14:39:00,2011-10-20 14:33:00,2011-12-09 12:50:00,NaN
UnitPrice,397884.0,3.116488,0.001,1.25,1.95,3.75,8142.75,22.097877
CustomerID,397884.0,15294.423453,12346.0,13969.0,15159.0,16795.0,18287.0,1713.14156


In [8]:
ds.columns

Index(['Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID',
       'Country'],
      dtype='object')

In [9]:
# Información base
print("Fecha mínima:", ds["InvoiceDate"].min())
print("Fecha máxima:", ds["InvoiceDate"].max())
print("Clientes únicos:", ds["CustomerID"].nunique())
print("Productos únicos:", ds["Description"].nunique())
print("Países:", ds["Country"].nunique())
print(f"Rango de fechas: {ds['InvoiceDate'].min().date()} → {ds['InvoiceDate'].max().date()}")


Fecha mínima: 2010-12-01 08:26:00
Fecha máxima: 2011-12-09 12:50:00
Clientes únicos: 4338
Productos únicos: 3877
Países: 37
Rango de fechas: 2010-12-01 → 2011-12-09


# Feature Engineering

In [10]:
feature_engineer = FeatureEngineer(ds)

In [11]:
df_engineered = feature_engineer.run()


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\src\app\train\feature_engineer.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(self.historial_compra)   # <-- ahora acepta g


In [12]:
df_engineered

,Description,InvoiceDate,Country,Quantity,Revenue,UnitPrice,CustomerID,n_past_invoices,prev_date,recency_days,spend_prior,qty_prior,avg_ticket_prior,avg_qty_per_invoice_prior,next_date,days_to_next,y_repurchase_30d
0,MEDIUM CERAMIC TOP STORAGE JAR,2011-01-18 10:01:00,United Kingdom,74215,77183.60,1.04,12346,0,NaT,9999,0.00,0,0.000000,0.000000,NaT,NaN,0
1,3D DOG PICTURE PLAYING CARDS,2010-12-07 14:57:00,Iceland,24,70.80,2.95,12347,0,NaT,9999,0.00,0,0.000000,0.000000,2010-12-07 14:57:00,0.0,1
2,AIRLINE BAG VINTAGE JET SET BROWN,2010-12-07 14:57:00,Iceland,4,17.00,4.25,12347,1,2010-12-07 14:57:00,0,70.80,24,70.800000,24.000000,2010-12-07 14:57:00,0.0,1
3,ALARM CLOCK BAKELIKE CHOCOLATE,2010-12-07 14:57:00,Iceland,4,15.00,3.75,12347,2,2010-12-07 14:57:00,0,87.80,28,43.900000,14.000000,2010-12-07 14:57:00,0.0,1
4,ALARM CLOCK BAKELIKE GREEN,2010-12-07 14:57:00,Iceland,4,15.00,3.75,12347,3,2010-12-07 14:57:00,0,102.80,32,34.266667,10.666667,2010-12-07 14:57:00,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
387655,SWISS CHALET TREE DECORATION,2011-10-12 10:23:00,United Kingdom,24,6.96,0.29,18287,63,2011-10-12 10:23:00,0,1739.84,1442,27.616508,22.888889,2011-10-12 10:23:00,0.0,1
387656,TREE T-LIGHT HOLDER WILLIE WINKIE,2011-10-12 10:23:00,United Kingdom,12,19.80,1.65,18287,64,2011-10-12 10:23:00,0,1746.80,1466,27.293750,22.906250,2011-10-28 09:29:00,15.0,1
387657,PAINTED METAL STAR WITH HOLLY BELLS,2011-10-28 09:29:00,United Kingdom,48,18.72,0.39,18287,65,2011-10-12 10:23:00,15,1766.60,1478,27.178462,22.738462,2011-10-28 09:29:00,0.0,1
387658,SET OF 3 WOODEN SLEIGH DECORATIONS,2011-10-28 09:29:00,United Kingdom,36,45.00,1.25,18287,66,2011-10-28 09:29:00,0,1785.32,1526,27.050303,23.121212,2011-10-28 09:29:00,0.0,1


# Modelando con MLFlow

In [13]:
import mlflow


experiment_name = "recompra-LogReg"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name)

# Enable autologging for sklearn models
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True,
    disable=False,
    exclusive=False,
    disable_for_unsupported_versions=False,
    silent=False,
    max_tuning_runs=5
)

In [14]:
num_feats = [
    'recency_days','n_past_invoices','spend_prior','qty_prior',
    'avg_ticket_prior','avg_qty_per_invoice_prior','UnitPrice','Quantity','Revenue'
]
cat_feats = ['Country']

In [15]:


model = LogisticRegression(max_iter=500)

# 3) Instancia y entrena
trainer = TrainMlflow(
    df=df_engineered,
    numeric_features=num_feats,
    categorical_features=cat_feats,
    target_column='y_repurchase_30d',
    model=model,
    mlflow_setup={"tracking_uri": "file:./mlruns", "experiment_name": "OnlineRetail"}
)

pipeline, run_id = trainer.train()
trainer.pipeline = pipeline                  # <- necesario para save_model()
trainer.save_model("models/model.pkl")       # ✅ Modelo guardado en models/model.pkl


Rango total: 2010-12-01 08:26:00 → 2011-12-09 12:50:00 | cutoff: 2011-11-09 12:50:00
train_end: 2011-09-01 00:00:00
train: 218546 | test: 102095
pos_rate train=0.972 | test=0.979


2025/09/20 17:12:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/09/20 17:12:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\Proyect

MLflow Run ID: d9d7c79ebdbd4e4ea2884c0b81e6d428
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 0.9720
Test Accuracy: 0.9786
🏃 View run rare-grub-51 at: http://127.0.0.1:5000/#/experiments/1/runs/d9d7c79ebdbd4e4ea2884c0b81e6d428
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
✅ Modelo guardado en models/model.pkl


'models/model.pkl'

In [16]:
mlflow.set_experiment("recompra-optuna")

params = {
            'solver': ('categorical', ['lbfgs', 'liblinear', 'saga']),
            'C':      ('float', 1e-3, 1e2, True),
            'max_iter': ('int', 300, 1500),
            'class_weight': ('categorical', [None, 'balanced']),
            # solver-specific penalties are tricky to encode generically—start simple with l2
            'penalty': ('categorical', ['l2']),
}

trainer = TrainOptuna(
    df=df_engineered,
    numeric_features=num_feats,
    categorical_features=cat_feats,
    target_column='y_repurchase_30d',
    model_class=LogisticRegression,
    model_params={},                 
    n_trials=30,                     
    optimization_metric='roc_auc',   
    param_distributions=params,
)

best_pipeline, best_run_id, study = trainer.train()   # runs Optuna + logs to MLflow
trainer.save_model("models/modeloptuna.pkl")


[I 2025-09-20 17:12:33,051] A new study created in memory with name: optuna_LogisticRegression
c:\Users\gabri\OneDrive\ProyectoFinalMLOps\src\app\train\train_mlflow_advance.py:247: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
2025/09/20 17:12:33 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8a74c9e5d24f48379717c90d19ab9dfe', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


Rango total: 2010-12-01 08:26:00 → 2011-12-09 12:50:00 | cutoff: 2011-11-09 12:50:00
train_end: 2011-09-01 00:00:00
train: 218546 | test: 102095
pos_rate train=0.972 | test=0.979
Starting Optuna optimization with 30 trials...
Optimizing for: roc_auc
Model type: LogisticRegression


2025/09/20 17:12:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/09/20 17:13:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\Proyect

🏃 View run treasured-boar-803 at: http://127.0.0.1:5000/#/experiments/2/runs/8a74c9e5d24f48379717c90d19ab9dfe
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
2025/09/20 17:13:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing 

...

c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
2025/09/20 17:42:01 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing 

🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/3/runs/8ca44392744a4103847391f843a871cf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3

Optimization complete!
Best roc_auc: 0.6931
Best parameters: {'solver': 'lbfgs', 'C': 0.0010582495531749523, 'max_iter': 1184, 'class_weight': 'balanced', 'penalty': 'l2'}


2025/09/20 17:42:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2025/09/20 17:42:08 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\gabri\OneDrive\Proyect


Best Model MLflow Run ID: 110275231d034c27a158c01dc387dfec
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 0.4479
Test Accuracy: 0.6210
🏃 View run best_model_LogisticRegression at: http://127.0.0.1:5000/#/experiments/3/runs/110275231d034c27a158c01dc387dfec
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
✅ Modelo guardado en models/modeloptuna.pkl


'models/modeloptuna.pkl'